## Functions

A **function** is a named, reusable block of code that performs a specific task.
Instead of writing the same logic in multiple places, you write it once in a function and call it by name.

### Anatomy of a Kotlin function

```
fun  functionName  (param: Type, ...)  : ReturnType  {  body  }
 ↑        ↑               ↑                  ↑            ↑
keyword  name        parameters        return type     logic
```

| Component | Required? | Notes |
|---|---|---|
| `fun` | Yes | Keyword that starts every function declaration |
| Name | Yes | Use camelCase verbs: `getUser`, `sendEmail`, `isValid` |
| Parameters | No | Comma-separated `name: Type` pairs in parentheses |
| Return type | No | Omit when the function returns nothing meaningful (`Unit`) |
| Body | Yes | Curly-brace block **or** single expression after `=` |

In [ ]:
fun functionName(/* param: Type */): Unit {
    // function body — the code that runs each time this function is called
}

// The compiler infers Unit automatically, so this is identical:
fun functionName2(/* param: Type */) {
    // omitting ": Unit" is the standard Kotlin style
}

## `Unit` — the "nothing meaningful to return" type

`Unit` is Kotlin's equivalent of Java's `void`. A function whose only purpose is to produce a **side effect**
(print something, update a UI, write to a database) returns `Unit`.

Key points:
- `Unit` is an actual type — it has exactly one value, also called `Unit`.
- You almost never need to write `: Unit` explicitly; the compiler infers it when you omit the return type.
- A function body that doesn't end with a `return <value>` statement automatically returns `Unit`.

```
Side effect = an observable action beyond computing a value
Examples: printing to console, writing to a file, sending a network request
```

In [ ]:
// This function's only job is a side effect (printing).
// It returns Unit — the compiler infers this; no need to write ": Unit".
fun printHello() {
    println("Hello, World!")
}

printHello()

// You can also store the Unit return value, though this is rarely useful:
val result: Unit = printHello()
println(result) // prints "kotlin.Unit"

## Parameters — a type analysis

### `Any`: the root of the Kotlin type hierarchy

Every type in Kotlin ultimately inherits from `Any`. Using `Any` as a parameter type means
"this function accepts a value of literally any type."

```
        Any
       /   \
    Int   String   Double   Boolean   YourClass   ...
```

**When `Any` seems useful:**
- You genuinely need to handle different types in one function (e.g., a generic logging utility).

**Why `Any` is usually a red flag:**
- Inside the function you can only call methods defined on `Any` itself: `equals()`, `hashCode()`, `toString()`.
- To do anything useful with the value you must cast it (`as Int`, `as String`) — which can crash at runtime.
- It defeats the compiler's type checking: passing the wrong type becomes a runtime error instead of a compile-time error.

> **Rule of thumb:** reach for `Any` only when you truly cannot be more specific. In practice, generics (`fun <T> process(value: T)`) are almost always the right tool instead.

In [ ]:
fun myFunction(param: Any) {
    // With Any, the only safe operations are defined on Any itself:
    println(param.toString())   // OK — Any has toString()
    println(param.hashCode())   // OK — Any has hashCode()

    // To do something type-specific, you must check and cast first:
    if (param is String) {
        println(param.uppercase())  // smart cast: compiler knows it's String here
    }
    if (param is Int) {
        println(param * 2)          // smart cast: compiler knows it's Int here
    }
    // Without these checks, param.uppercase() would be a compile error.
}

myFunction("hello")
myFunction(42)

## Prefer specific types (or interfaces) for parameters

Declaring the exact type you expect gives you three concrete benefits:

1. **Compiler safety** — passing a wrong type is caught at compile time, before the app ever runs.
2. **Full API access** — you can call every method of that type directly, no casting needed.
3. **Self-documenting code** — the signature tells the reader exactly what kind of value is expected.

Using an **interface** instead of a concrete class is even better when you want flexibility:
```kotlin
fun displayLength(input: CharSequence) { ... }
//  ↑ accepts String, StringBuilder, and any other CharSequence — flexible but still type-safe
```

This is one of the core ideas behind the **Dependency Inversion Principle** (SOLID — D).

In [ ]:
fun processUserInput(input: String) {
    // Because the type is String, all String methods are available immediately — no cast needed.
    println("Original : $input")
    println("Trimmed  : ${input.trim()}")
    println("Uppercase: ${input.uppercase()}")
    println("Length   : ${input.length}")
    // If someone passes an Int by mistake, the compiler rejects it before the app runs.
}

processUserInput("  hello, world!  ")

## Default Parameter Values & Named Arguments

Kotlin lets you assign a default value to any parameter. If the caller omits that argument, the default is used.

### Why this matters
In Java, you'd create multiple overloaded versions of the same function to handle optional parameters.
Kotlin replaces all of those with **one function** and default values — less code, same flexibility.

```java
// Java — three overloads to simulate one optional parameter
String format(String number) { return format(number, "+56"); }
String format(String number, String code) { return code + " " + number; }
```
```kotlin
// Kotlin — one function covers all cases
fun format(number: String, countryCode: String = "+56") = "$countryCode $number"
```

### Named arguments
You can pass arguments **by name** instead of by position. This:
- Makes call sites self-documenting (`countryCode = "+57"` is clearer than just `"+57"`)
- Lets you skip to a specific parameter when earlier ones have defaults
- Eliminates confusion when multiple parameters share the same type

In [ ]:
fun phoneNumberProcess(number: String, countryCode: String = "+56"): String {
    return "$countryCode $number"
}

// 1. Only required param — countryCode defaults to "+56"
println(phoneNumberProcess("123-456-7890"))

// 2. Both params by position — order must match the declaration
println(phoneNumberProcess("123-456-7890", "+57"))

// 3. Second param by name — position still correct but intent is explicit
println(phoneNumberProcess("123-456-7890", countryCode = "+57"))

// 4. All params by name — order no longer matters
println(phoneNumberProcess(number = "123-456-7890", countryCode = "+57"))

// Named args shine when a function has many parameters with defaults —
// you only specify what you actually need to change.

# Expression Body

When a function consists of **a single expression**, you can replace the curly-brace block and `return` statement with `=` followed by that expression.

```kotlin
// Block body (always valid)
fun square(x: Int): Int {
    return x * x
}

// Expression body (concise alternative for single-expression functions)
fun square(x: Int): Int = x * x

// Type inference: the compiler can often infer the return type from the expression
fun square(x: Int) = x * x
```

### When to use each style
| Style | Use when |
|---|---|
| Expression body `=` | The entire logic is one expression; the type is obvious |
| Block body `{ }` | Multiple statements, early returns, or complex conditional logic |

> **Common gotcha:** a block body that does NOT have a `return` statement returns `Unit`, even if the last line computes a value. See the next cell for a live example.

In [ ]:
// ✅ Expression body: the result of (x * x) IS the return value
fun powerOfTwo(x: Int) = x * x

// ❌ Block body WITHOUT return: x * x is computed but immediately discarded.
//    The function implicitly returns Unit — not the computed value!
fun powerOfTwoV2(x: Int) {
    x * x   // ← this result goes nowhere; no return statement
}

println(powerOfTwo(5))    // 25   — correct, returns Int
println(powerOfTwoV2(5))  // kotlin.Unit — the forgotten return statement trap

// Fix: add return, or convert to expression body
fun powerOfTwoV3(x: Int): Int {
    return x * x   // ✅ explicit return
}

In [ ]:
// LOCAL (inner) functions — functions defined inside other functions

fun myFunction() {
    val myNumber = 15

    // innerFunction is only visible inside myFunction — it cannot be called from outside.
    // It "closes over" myNumber from the enclosing scope — this is called a CLOSURE.
    fun innerFunction() {
        println("The number is ${myNumber.plus(5)}")  // reads myNumber from the outer scope
    }

    innerFunction()
}

myFunction()

// Local functions with expression bodies can also capture outer variables:
fun functionV2() {
    val myNumber = 15

    fun doubleNumber() = myNumber * 2  // captures myNumber — no parameter needed

    println(doubleNumber())  // 30
}

functionV2()

// Use local functions when a helper is needed only in one place.
// They keep the helper out of the class-level API while still being readable.

## Closures — capturing variables from the enclosing scope

A **closure** is a function (or lambda) that "remembers" variables from the scope where it was defined,
even after execution has moved on past that scope.

In Kotlin, closures can **read and modify** `var` variables from the enclosing scope.
This is different from Java, where lambdas can only capture *effectively final* variables.

```
outer scope defines: var counter = 0
         ↓
  function captures counter   ← closure
         ↓
  function modifies counter   ← mutation through the closure
```

> **Watch out:** functions that silently mutate external state can make code hard to test and reason about.
> Prefer returning a new value over modifying a captured variable when possible.

In [ ]:
var counter = 0

// incrementCounter is a regular function that MODIFIES the outer var directly (closure)
fun incrementCounter() {
    counter = counter + 1
}

println(counter)     // 0 — initial value
incrementCounter()
println(counter)     // 1 — counter was mutated by the function

// ⚠️ TRICKY: incrementCounterV2 uses expression body with a lambda literal { ... }
// This means it RETURNS a lambda — it does NOT execute it!
fun incrementCounterV2() = { counter = counter + 1 }

// incrementCounterV2() returns a () -> Unit lambda; the lambda body does NOT run yet.
// To actually run the lambda you would call: incrementCounterV2()()
//                                                              ↑↑ invokes the returned lambda
val doIncrement = incrementCounterV2()  // stores the lambda
doIncrement()                           // now the lambda runs and counter becomes 2
println(counter)                        // 2

# Functions as First-Class Citizens

In Kotlin, functions are **first-class citizens** — they are values, just like `Int` or `String`.
This means you can:

| Operation | Example |
|---|---|
| Store a function in a variable | `val fn = ::myFunction` |
| Pass a function as a parameter | `process(5, ::double)` |
| Return a function from a function | `fun makeAdder(n: Int): (Int) -> Int = { it + n }` |

### Function types
Every function has a **type** expressed as `(ParameterTypes) -> ReturnType`:

```kotlin
(Int) -> Int        // takes one Int, returns an Int
(String) -> Unit    // takes a String, returns nothing (a callback)
() -> Boolean       // takes nothing, returns a Boolean
(Int, Int) -> Int   // takes two Ints, returns an Int
```

### Higher-order functions
A function that **accepts another function** (or returns one) is called a **higher-order function**.
This is the foundation of Kotlin's collection APIs (`map`, `filter`, `forEach`) and Android's
click-listener pattern (`setOnClickListener { ... }`).

In [ ]:
// Higher-order function: accepts a function as a parameter
// The type (Int) -> Int means "a function that takes an Int and returns an Int"
fun numberInputProcessor(number: Int, processor: (Int) -> Int): Int {
    return processor(number)  // calls the function that was passed in
}

fun double(x: Int): Int = x * 2
fun applyDiscount10Percent(price: Int) = price - (price * 10 / 100)

// :: is the function reference operator — it turns a named function into a value
val processor1: (Int) -> Int = ::double   // "give me double as a value"

println("5 doubled          : ${numberInputProcessor(5, processor1)}")         // 10
println("100 after 10% off  : ${numberInputProcessor(100, ::applyDiscount10Percent)}") // 90

// You can also pass a lambda directly (anonymous function inline):
println("5 tripled          : ${numberInputProcessor(5, { x -> x * 3 })}")    // 15

In [ ]:
// CALLBACK PATTERN — a function accepts another function to call when something happens.
// This is the same pattern Android uses for click listeners, coroutine callbacks, etc.

fun Button(text: String, onClick: () -> Unit) {
    // onClick has type () -> Unit: takes nothing, returns nothing — a pure side-effect callback
    println("Button '$text' created")
    println("Simulating click...")
    onClick()  // invoke whatever behaviour the caller provided
    println()
}

fun printClick()    = println("Button clicked!")
fun loginAfterClick() = println("Logging in...")

// Pass named functions as callbacks using ::
Button("Click me", onClick = ::printClick)
Button("Login",    onClick = ::loginAfterClick)

// The caller decides what happens on click — the Button doesn't need to know.
// In Android: view.setOnClickListener(::handleClick) follows the exact same pattern.

## Trailing Lambda Syntax

When the **last parameter** of a function is a function type, Kotlin lets you move the lambda **outside** the parentheses. This is called *trailing lambda* syntax and is purely stylistic sugar — both forms compile to the same bytecode.

```kotlin
// Normal call — lambda inside parentheses
Button("Click me", { println("clicked") })

// Trailing lambda — lambda moved outside parentheses
Button("Click me") { println("clicked") }
```

If the lambda is the **only** argument, you can drop the parentheses entirely:
```kotlin
listOf(1, 2, 3).forEach { println(it) }
//              ↑ no parentheses — the lambda IS the only argument
```

This is why Kotlin's collection operations (`map`, `filter`, `sortedBy`) and Android's
`setOnClickListener { }` feel like built-in language constructs even though they're just functions.

In [ ]:
// onClick has a default value — the lambda { println("Default Click") }
fun ButtonV2(text: String, onClick: () -> Unit = { println("Default Click") }) {
    println("Button '$text' created")
    println("Simulating click...")
    onClick()
    println()
}

// 1. No lambda supplied — the default runs
ButtonV2("Click me")

// 2. Trailing lambda — the lambda is the last (and only non-default) argument,
//    so it moves outside the parentheses. Same as: ButtonV2("...", { ... })
ButtonV2("Trailing Click") {
    println("Trailing Lambda Clicked!")
}

// 3. The two forms below are exactly equivalent:
ButtonV2("A", { println("inline") })   // lambda inside parens
ButtonV2("B") { println("trailing") }  // trailing lambda — preferred style

# The `it` Implicit Parameter

When a lambda takes **exactly one parameter**, Kotlin lets you omit the parameter declaration entirely
and refer to it as `it` — the implicit name.

```kotlin
// Explicit parameter name:
val double: (Int) -> Int = { number -> number * 2 }

// Using it (identical, just shorter):
val double: (Int) -> Int = { it * 2 }
```

### When to use `it` vs. a named parameter

| Situation | Recommendation |
|---|---|
| Short, obvious transformation | `it` is fine — `list.map { it * 2 }` |
| Nested lambdas | **Name the parameter** — `it` becomes ambiguous |
| Complex logic | **Name the parameter** — `user -> user.age > 18` reads better than `it.age > 18` |

> **Rule:** if reading `it` requires looking up what "it" refers to, give it a name.

In [ ]:
// apply10Percent is a function that RETURNS another function (a lambda).
// Its return type is (Int) -> Int — a function from Int to Int.
// This technique is called "partial application" or a "factory function for functions."
fun apply10Percent(product: Int): (Int) -> Int = { product - (product * 10 / 100) }
//                                                 ↑ this lambda captures `product` from the outer scope

// calculateTotalPrice accepts a price AND a discount function to apply
fun calculateTotalPrice(price: Int, discount: (Int) -> Int): Int {
    return discount(price)
}

// apply10Percent is a reference to the function (not a call to it yet)
// When called with an Int, it returns the lambda that computes the discounted price
val total = 100
println("Total price after discount: ${calculateTotalPrice(total, apply10Percent)}")
// apply10Percent is passed as the discount strategy — calculateTotalPrice calls it internally

In [ ]:
data class Product(val name: String, val price: Int)

val products = listOf(
    Product("Laptop",      1000),
    Product("Smartphone",   500),
    Product("Tablet",       300)
)

// Lambda with an EXPLICIT parameter name (currentProduct) instead of it.
// Use a name when the lambda is long enough that "it" would be confusing.
val discountProduct: (Product) -> Int = { currentProduct -> currentProduct.price }

// Using it (shorter, fine for one-liners where the type is obvious from context):
val getPrice: (Product) -> Int = { it.price }

println("Prices via named param : ${products.map(discountProduct)}")
println("Prices via it          : ${products.map(getPrice)}")

// Practical use — filter and transform with collection functions:
val affordableNames = products
    .filter  { it.price <= 500 }          // it = Product — short, clear
    .map     { product -> product.name }  // named param — reads nicely in a chain
println("Affordable products: $affordableNames")